# 📊 Telecom Customer Churn — Cleaning, EDA & Model Training

This notebook walks through the full workflow for the churn-prediction project:

1. Load the raw dataset
2. Explore it (EDA + visualizations)
3. Build a reusable cleaning/preprocessing pipeline
4. Split into train / validation / test sets
5. Train four classification models
6. Compare their performance (metrics, ROC curves, feature importance, confusion matrices)
7. Export the trained models for serving via API

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)

import joblib

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
logging.getLogger(__name__)

%matplotlib inline

## 2. Load the dataset

In [ ]:
data_path = Path("./data/dataset-telecom.csv")
df = None

try:
    if data_path.exists():
        logging.info("data loading ...")
        df = pd.read_csv(data_path)
    else:
        raise FileNotFoundError("file not found")
except FileNotFoundError as e:
    logging.warning(f"{e} : {data_path}")

logging.info("data loading completed")
df = df.drop(columns=["customerID", "TotalCharges"], errors="ignore")
df.head()

## 3. First look at the data

Basic shape, types, and missing-value check before doing anything else.

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing = df.isna().sum()
missing[missing > 0]

## 4. Exploratory Data Analysis

A few views of churn against the features that usually matter most:
contract type, tenure, monthly charges, and internet service.

### 4.1 Overall churn rate

In [ ]:
churn_counts = df["Churn"].value_counts()

fig, ax = plt.subplots()
sns.barplot(x=churn_counts.index, y=churn_counts.values, ax=ax, palette="Set2")
ax.set_title("Customer churn distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of customers")
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 20, str(v), ha="center")
plt.show()

print(f"Churn rate: {churn_counts.get('Yes', 0) / len(df):.1%}")

### 4.2 Churn by contract type

In [ ]:
fig, ax = plt.subplots()
sns.countplot(data=df, x="Contract", hue="Churn", ax=ax, palette="Set2")
ax.set_title("Churn by contract type")
plt.show()

### 4.3 Monthly charges & tenure vs churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df, x="MonthlyCharges", hue="Churn", fill=True, ax=axes[0])
axes[0].set_title("Monthly charges by churn")

sns.kdeplot(data=df, x="tenure", hue="Churn", fill=True, ax=axes[1])
axes[1].set_title("Tenure (months) by churn")

plt.tight_layout()
plt.show()

### 4.4 Churn by internet service

In [ ]:
fig, ax = plt.subplots()
sns.countplot(data=df, x="InternetService", hue="Churn", ax=ax, palette="Set2")
ax.set_title("Churn by internet service type")
plt.show()

### 4.5 Correlation heatmap

Numeric + binary-encoded columns only. `Churn` and the Yes/No service
columns are mapped to 0/1 just for this plot so they can be included.

In [ ]:
corr_df = df.copy()

binary_like = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
for col in binary_like:
    corr_df[col] = corr_df[col].map({"Yes": 1, "No": 0})

corr_df["SeniorCitizen"] = corr_df["SeniorCitizen"].astype(int)

numeric_for_corr = corr_df[["tenure", "MonthlyCharges", "SeniorCitizen"] + binary_like]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(numeric_for_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
ax.set_title("Feature correlation heatmap")
plt.show()

## 5. Cleaning & preprocessing pipeline

Drops unused columns, then builds a `ColumnTransformer` with three branches:

- **Numeric** (`MonthlyCharges`, `tenure`) → median-impute, then `StandardScaler`
- **Categorical** (`gender`, `InternetService`, `Contract`, `PaymentMethod`) → most-frequent-impute, then one-hot encode (`drop='first'` to avoid the dummy-variable trap)
- **Yes/No-style columns** → most-frequent-impute, then a custom cleaner that collapses
  `"No internet service"` / `"No phone service"` into `"No"` and maps `Yes/No` to `1/0`
- `SeniorCitizen` passes through unchanged (already 0/1)

In [ ]:
def clean_no_service(col):
    df_col = pd.DataFrame(col).replace({
        "No internet service": 0,
        "No phone service": 0,
        "No": 0,
        "Yes": 1
    })
    # Explicitly cast to int to avoid downcasting warnings
    return df_col.astype(int).to_numpy()



Cleaner = FunctionTransformer(clean_no_service)

In [ ]:
numeric_columns = ["MonthlyCharges", "tenure"]
categorical_columns = ["gender", "InternetService", "Contract", "PaymentMethod"]
convertToNumeric_columns = [
    "Partner", "StreamingMovies", "Dependents", "MultipleLines", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "PaperlessBilling", "Churn", "PhoneService",
]

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first")),
])

convertToNumeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("cleaner", Cleaner),
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns),
    ("conv", convertToNumeric_pipeline, convertToNumeric_columns),
    ("senior", "passthrough", ["SeniorCitizen"]),
])

data_cleaned_array = preprocessor.fit_transform(df)

In [ ]:
feature_names = []
feature_names.extend(preprocessor.named_transformers_["num"].get_feature_names_out(numeric_columns))
feature_names.extend(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_columns))
feature_names.extend(convertToNumeric_columns)  # conv pipeline keeps original names
feature_names.append("SeniorCitizen")

data_cleaned = pd.DataFrame(data_cleaned_array, columns=feature_names)

print(data_cleaned.shape)
data_cleaned.head()

In [ ]:
save_cleaned_data = Path("./data/data_cleaned.csv")
save_cleaned_data.parent.mkdir(parents=True, exist_ok=True)
data_cleaned.to_csv(save_cleaned_data, index=False)

logging.info("cleaned dataset saved as csv file")

## 6. Train / validation / test split

70% train, 15% validation, 15% test.

In [ ]:
y_data = data_cleaned["Churn"]
X_data = data_cleaned.drop(columns=["Churn"])

X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.3, random_state=10
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

## 7. Model training & evaluation

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_val, y_val, X_test, y_test, proba=False):
    """Train, predict, and print metrics for a given model."""
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    print(f"\n{name} - Training Performance:")
    print(classification_report(y_train, y_pred_train))

    print(f"\n{name} - Validation Performance:")
    print(classification_report(y_val, y_pred_val))
    print("Confusion Matrix (Validation):")
    print(confusion_matrix(y_val, y_pred_val))

    print(f"{name} - Test Performance:")
    print(classification_report(y_test, y_pred_test))
    print("Confusion Matrix (Test):")
    print(confusion_matrix(y_test, y_pred_test))

    if proba:
        y_prob = model.predict_proba(X_test)[:, 1]
        print(f"ROC AUC (Test): {roc_auc_score(y_test, y_prob):.3f}")

In [ ]:
log_reg = LogisticRegression(max_iter=5000, solver="saga", class_weight="balanced")
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10,
    min_samples_split=20, min_samples_leaf=10,
    random_state=42, n_jobs=-1
)

# Voting classifier combining all three — soft voting averages predicted probabilities
voting_clf = VotingClassifier(
    estimators=[("lr", log_reg), ("rf", rf), ("gb", gb)],
    voting="soft",
)

In [ ]:
evaluate_model("Logistic Regression", log_reg, X_train, y_train, X_val, y_val, X_test, y_test, proba=True)

In [ ]:
evaluate_model("Gradient Boosting", gb, X_train, y_train, X_val, y_val, X_test, y_test, proba=True)

In [ ]:
evaluate_model("Random Forest", rf, X_train, y_train, X_val, y_val, X_test, y_test, proba=True)

In [ ]:
evaluate_model("Voting Classifier", voting_clf, X_train, y_train, X_val, y_val, X_test, y_test, proba=True)

## 8. Compare models side by side

Pull the test-set metrics for all four models into one table and chart,
rather than reading them out of scattered `classification_report` blocks.

In [ ]:
trained_models = {
    "Logistic Regression": log_reg,
    "Gradient Boosting": gb,
    "Random Forest": rf,
    "Voting Classifier": voting_clf,
}

rows = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })

comparison_df = pd.DataFrame(rows).set_index("model")
comparison_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.plot(kind="bar", ax=ax)
ax.set_title("Model comparison — test set metrics")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. ROC curves

All four models plotted together — a model that hugs the top-left corner
(closer to AUC = 1.0) separates churners from non-churners better.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — all models")
ax.legend(loc="lower right")
plt.show()

## 10. Confusion matrices

Side-by-side heatmaps on the test set for a quick visual read of
false positives vs false negatives per model.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 11. Feature importance (tree-based models)

Random Forest and Gradient Boosting expose `feature_importances_` directly —
useful for sanity-checking against the correlation heatmap from section 4.5.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, model) in zip(axes, [("Random Forest", rf), ("Gradient Boosting", gb)]):
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    importances = importances.sort_values(ascending=True).tail(12)
    importances.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"Top features — {name}")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()

## 12. Save trained models

Saved with `joblib` (the standard choice for sklearn estimators — handles
large numpy arrays inside the model more efficiently than plain `pickle`).

In [ ]:
models = [
    {"name": "logistic_regression.pkl", "model": log_reg},
    {"name": "gradient_boosting.pkl", "model": gb},
    {"name": "voting_classifier.pkl", "model": voting_clf},
    {"name": "random_forest.pkl", "model": rf},
]

Path("./models").mkdir(parents=True, exist_ok=True)
for mod in models:
    joblib.dump(mod["model"], "./models/" + mod["name"])

print("Saved:", [m["name"] for m in models])

## 13. Summary

- Cleaned and encoded 19 raw features into a model-ready feature set (`data_cleaned.csv`)
- Trained four classifiers: Logistic Regression, Gradient Boosting, Random Forest, and a soft-voting ensemble
- Compared them on accuracy, precision, recall, F1, and ROC AUC on a held-out test set
- Exported all four as `.pkl` files under `./models/` for serving behind an API

**Next steps:** tune hyperparameters (e.g. `GridSearchCV`), try threshold tuning instead of the default 0.5
cutoff to improve recall on the churn class, and re-evaluate periodically as new customer data comes in.